# Notebook 00b — Cycle Plots

One plot per complete cracking cycle. Loads clean data from Delta table,
then for each run plots the full cycle (decoke→decoke) with all signals.

**Panels per run:**
1. Feed total
2. COT (coil outlet temperature)
3. dd_abs_max (worst tube deviation across whole furnace)
4. dd_abs_max per pass (A/B/C/D)


## 1. Parameters


In [0]:
import os, sys, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

warnings.filterwarnings('ignore')
plt.rcParams.update({
    'figure.dpi': 120,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'grid.alpha': 0.3,
    'font.size': 10,
})

REPO_ROOT = os.path.dirname(os.path.dirname(os.path.abspath('.')))
sys.path.insert(0, REPO_ROOT)

# Load .env for local VSCode runs
_env = os.path.join(REPO_ROOT, '.env')
if os.path.exists(_env):
    with open(_env) as _f:
        for _line in _f:
            _line = _line.strip()
            if _line and not _line.startswith('#') and '=' in _line:
                _k, _, _v = _line.partition('=')
                os.environ.setdefault(_k.strip(), _v.strip())

from olefins_ddf.io_events import get_spark

FURNACE       = '1HA'
DELTA_CATALOG = 'indorama_corporate_olefins_paas_azure_weu_dev_research.workspaces'
OUTPUT_DIR    = os.path.join(REPO_ROOT, 'output', FURNACE, 'cycle_plots')

os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f'Furnace    : {FURNACE}')
print(f'Output dir : {OUTPUT_DIR}')


## 2. Load data from Delta


In [0]:
# Reset spark to Serverless in case a previous run left it bound to a remote cluster
from databricks.connect import DatabricksSession
try:
    spark.stop()
except Exception:
    pass
spark = DatabricksSession.builder.serverless(True).getOrCreate()

# Full feature matrix (includes warmup + tail context)
feat = spark.table(f'{DELTA_CATALOG}.{FURNACE.lower()}_features').toPandas()
ts_col = next((c for c in ['timestamp', 'ts'] if c in feat.columns), None)
feat = feat.set_index(ts_col)
feat.index = pd.to_datetime(feat.index)
feat.index.name = 'timestamp'

# Clean data (has run_id column)
clean = spark.table(f'{DELTA_CATALOG}.{FURNACE.lower()}_clean').toPandas()
ts_col = next((c for c in ['timestamp', 'ts'] if c in clean.columns), None)
clean = clean.set_index(ts_col)
clean.index = pd.to_datetime(clean.index)
clean.index.name = 'timestamp'

# Run windows from clean data
win_path = os.path.join(REPO_ROOT, 'output', FURNACE, '00b_clean', 'run_analysis_windows.csv')
if os.path.exists(win_path):
    win_df = pd.read_csv(win_path, parse_dates=['run_start','run_end','analysis_start','analysis_end'])
else:
    # Reconstruct from clean data if CSV not available
    win_df = (clean.groupby('run_id')
              .agg(analysis_start=('run_id', lambda x: x.index.min()),
                   analysis_end  =('run_id', lambda x: x.index.max()))
              .reset_index()
              .rename(columns={'run_id': 'run'}))
    win_df['run_start'] = win_df['analysis_start']
    win_df['run_end']   = win_df['analysis_end']

run_ids = sorted(clean['run_id'].unique())
print(f'Feature matrix : {feat.shape[0]:,} rows')
print(f'Clean data     : {clean.shape[0]:,} rows')
print(f'Complete runs  : {len(run_ids)} — {run_ids}')


## 3. Plot each cycle

One figure per run. Green = clean analysis window, red = warmup/tail.
4 panels: Feed · COT · dd_abs_max (furnace) · dd_abs_max per pass (A/B/C/D).


In [0]:
PASS_COLORS = {'A': '#1f77b4', 'B': '#ff7f0e', 'C': '#2ca02c', 'D': '#d62728'}

def plot_cycle(run_id, feat_full, clean_df, win_df, output_dir, furnace):
    """Plot one complete cracking cycle (decoke to decoke)."""

    # Get analysis window boundaries
    w = win_df[win_df['run'] == run_id]
    if w.empty:
        # Fallback: use clean data bounds
        run_clean = clean_df[clean_df['run_id'] == run_id]
        analysis_start = run_clean.index.min()
        analysis_end   = run_clean.index.max()
        run_start = analysis_start
        run_end   = analysis_end
    else:
        w = w.iloc[0]
        run_start      = pd.Timestamp(w['run_start'])
        run_end        = pd.Timestamp(w['run_end'])
        analysis_start = pd.Timestamp(w['analysis_start'])
        analysis_end   = pd.Timestamp(w['analysis_end'])

    # Slice full data for this run window (includes warmup + tail)
    seg = feat_full.loc[run_start:run_end]
    seg_clean = clean_df[clean_df['run_id'] == run_id]

    if seg.empty:
        print(f'  Run {run_id}: no data, skipping')
        return

    # Detect available pass columns
    pass_cols = {p: f'dd_abs_max_{p}' for p in 'ABCD'
                 if f'dd_abs_max_{p}' in feat_full.columns}

    n_panels = 4 if pass_cols else 3
    fig, axes = plt.subplots(n_panels, 1, figsize=(14, 3.2 * n_panels), sharex=True)

    # Clean window duration
    clean_days = (analysis_end - analysis_start).total_seconds() / 86400
    total_days = (run_end - run_start).total_seconds() / 86400

    fig.suptitle(
        f'{furnace}  |  Run {run_id}  |  '
        f'{run_start.date()} → {run_end.date()}  '
        f'({total_days:.1f} d total · {clean_days:.1f} d clean)',
        fontsize=11, fontweight='bold'
    )

    # Shade regions
    def shade(ax):
        ax.axvspan(run_start,      analysis_start, alpha=0.12, color='#e24b4a', zorder=0, label='warmup')
        ax.axvspan(analysis_start, analysis_end,   alpha=0.10, color='#1d9e75', zorder=0, label='clean')
        ax.axvspan(analysis_end,   run_end,        alpha=0.12, color='#e24b4a', zorder=0, label='tail')

    # Panel 1: Feed
    ax = axes[0]
    if 'feed_total' in seg.columns:
        ax.plot(seg.index, seg['feed_total'], color='#378add', lw=0.8, alpha=0.7, label='raw')
        ax.plot(seg_clean.index, seg_clean['feed_total'], color='#1d9e75', lw=1.2, label='clean')
    shade(ax)
    ax.set_ylabel('Feed total (NM³/H)')
    ax.legend(fontsize=7, loc='upper right')

    # Panel 2: COT
    ax = axes[1]
    cot_col = next((c for c in ['cot', 'cot_ctrl'] if c in seg.columns), None)
    if cot_col:
        ax.plot(seg.index, seg[cot_col], color='#7f77dd', lw=0.8, alpha=0.7, label='raw')
        ax.plot(seg_clean.index, seg_clean[cot_col], color='#085041', lw=1.2, label='clean')
    shade(ax)
    ax.set_ylabel('COT (°C)')
    ax.legend(fontsize=7, loc='upper right')

    # Panel 3: dd_abs_max (furnace-level)
    ax = axes[2]
    if 'dd_abs_max' in seg.columns:
        ax.plot(seg.index, seg['dd_abs_max'], color='#ef9f27', lw=0.8, alpha=0.5, label='raw')
        ax.plot(seg_clean.index, seg_clean['dd_abs_max'], color='#d85a30', lw=1.2, label='clean')
        ax.axhline(45, color='#e24b4a', ls='--', lw=1.0, label='Alarm 45°C')
        ax.axhline(30, color='#ef9f27', ls='--', lw=1.0, label='Pre-alarm 30°C')
    shade(ax)
    ax.set_ylabel('dd_abs_max (°C)')
    ax.legend(fontsize=7, loc='upper right')

    # Panel 4: dd_abs_max per pass
    if pass_cols and n_panels == 4:
        ax = axes[3]
        for p, col in pass_cols.items():
            color = PASS_COLORS.get(p, 'grey')
            ax.plot(seg.index, seg[col], lw=0.6, alpha=0.4, color=color)
            ax.plot(seg_clean.index, seg_clean[col], lw=1.0, color=color, label=f'Pass {p}')
        ax.axhline(45, color='#e24b4a', ls='--', lw=0.8)
        ax.axhline(30, color='#ef9f27', ls='--', lw=0.8)
        shade(ax)
        ax.set_ylabel('dd_abs_max\nper pass (°C)')
        ax.legend(fontsize=7, loc='upper right', ncol=2)

    axes[-1].set_xlabel('Date')
    plt.tight_layout()

    path = os.path.join(output_dir, f'{furnace}_run{run_id:02d}.png')
    plt.savefig(path, bbox_inches='tight')
    plt.show()
    plt.close()
    print(f'  Run {run_id:2d} saved → {path}')


print(f'Generating {len(run_ids)} cycle plots...')
for rid in run_ids:
    plot_cycle(rid, feat, clean, win_df, OUTPUT_DIR, FURNACE)

print(f'\nDone. All plots in: {OUTPUT_DIR}')
